In [6]:
import pandas as pd
import ir_datasets
from sklearn.metrics import cohen_kappa_score, roc_auc_score
from src.data import DATA_DIR_PROCESSED
import os

# Experiment 2: How good are LLM judgments on this dataset?

In [7]:
def merge_qrels(qrels, qrels_gen):
    # merge generated with original qrels
    qrels = qrels.merge(qrels_gen[["query_id", "doc_id", "relevance"]], on=[
                        "query_id", "doc_id"], how="left", suffixes=('_orig', '_gen'))

    # drop na valuse: qrels that were not sampled
    qrels = qrels.dropna(subset=["relevance_gen"])

    # some documents were not judged in the generated qrels
    # print(f"Number of missing judgements in generated qrels: {qrels[qrels['relevance_gen'] == 999].shape[0]} of {len(qrels)}")
    missing = qrels[qrels['relevance_gen'] == 999].shape[0]
    qrels = qrels[qrels['relevance_gen'] != 999]

    # Creqate binary relevance labels
    qrels["relevance_gen"] = qrels["relevance_gen"].astype(int)
    qrels["relevance_gen_bin"] = qrels["relevance_gen"].apply(
        lambda x: 0 if x > 0 else 1)
    qrels["relevance_orig_bin"] = qrels["relevance_orig"].apply(
        lambda x: 0 if x > 0 else 1)

    return qrels, missing

In [8]:
# load original qrels for reference
dataset = ir_datasets.load("disks45/nocr/trec-robust-2004")
qrels = pd.DataFrame(dataset.qrels)
qrels["query_id"] = qrels["query_id"].astype("category")
qrels["doc_id"] = qrels["doc_id"].astype("category")

# load generated qrels
BASE_DIR = DATA_DIR_PROCESSED / "exp2"

generated_qrels = []
for result_file in os.listdir(BASE_DIR):
    # Load qrels
    qrels_gen = pd.read_csv(os.path.join(BASE_DIR, result_file), sep=" ", names=["query_id", "iteration", "doc_id", "relevance"], dtype={
                            "query_id": pd.CategoricalDtype(), "doc_id": pd.CategoricalDtype(), "relevance": float, "iteration": int})

    qrels_merged, missing = merge_qrels(qrels, qrels_gen)

    # parse name
    task, model, prompt, topic, k = result_file.strip(".csv.gz").split("_")

    # bootstrap kappa
    binary, graded = [], []
    for _ in range(20):
        qrels_sampled = qrels_merged.sample(
            len(qrels_merged), random_state=None, replace=True)
        binary_ck = cohen_kappa_score(
            qrels_sampled["relevance_orig_bin"], qrels_sampled["relevance_gen_bin"])
        graded_ck = cohen_kappa_score(
            qrels_sampled["relevance_orig"], qrels_sampled["relevance_gen"])
        binary.append(binary_ck)
        graded.append(graded_ck)

    # 95 confidence interval
    binary_ci = (pd.Series(binary).quantile(0.975) -
                 pd.Series(binary).quantile(0.025)) / 2
    graded_ci = (pd.Series(graded).quantile(0.975) -
                 pd.Series(graded).quantile(0.025)) / 2

    # base
    binary_ck = cohen_kappa_score(
        qrels_merged["relevance_orig_bin"], qrels_merged["relevance_gen_bin"])
    graded_ck = cohen_kappa_score(
        qrels_merged["relevance_orig"], qrels_merged["relevance_gen"])

    # store results
    generated_qrels.append({
        "model": model,
        "topic": topic.replace("topics-", ""),
        "prompt": prompt,
        "missing": missing,
        "measure": "cohen_kappa_binary",
        "score": binary_ck,
        "confidence_95": binary_ci
    })

    generated_qrels.append({
        "model": model,
        "topic": topic.replace("topics-", ""),
        "prompt": prompt,
        "missing": missing,
        "measure": "cohen_kappa_graded",
        "score": graded_ck,
        "confidence_95": graded_ci
    })

    # MAE
    graded_mae = (qrels_merged["relevance_orig"] -
                  qrels_merged["relevance_gen"]).abs().mean()
    binary_mae = (qrels_merged["relevance_orig_bin"] -
                  qrels_merged["relevance_gen_bin"]).abs().mean()
    generated_qrels.append({
        "model": model,
        "topic": topic.replace("topics-", ""),
        "prompt": prompt,
        "missing": missing,
        "measure": "mae_binary",
        "score": binary_mae,
        "confidence_95": None
    })
    generated_qrels.append({
        "model": model,
        "topic": topic.replace("topics-", ""),
        "prompt": prompt,
        "missing": missing,
        "measure": "mae_graded",
        "score": graded_mae,
        "confidence_95": None
    })

    # # AUC
    # binary_auc = roc_auc_score(
    #     qrels_merged["relevance_orig_bin"], qrels_merged["relevance_gen_bin"])
    # # graded_auc = roc_auc_score(qrels_merged["relevance_orig"], qrels_merged["relevance_gen"])
    # generated_qrels.append({
    #     "model": model,
    #     "topic": topic.replace("topics-", ""),
    #     "prompt": prompt,
    #     "missing": missing,
    #     "measure": "binary_auc",
    #     "score": graded_mae,
    #     "confidence_95": None
    # })

In [9]:
df = pd.DataFrame(generated_qrels)

In [10]:
df.sort_values(by=["measure", "model", "topic"])

,model,topic,prompt,missing,measure,score,confidence_95
16,gpt-4.1,trec,robust-DNA-zero,0,cohen_kappa_binary,0.643923,0.027793
0,qwen3-14B-MT100-no-think,trec,robust-DNA-zero-shot-masked-description-narrative,0,cohen_kappa_binary,0.636157,0.021624
4,qwen3-14B-MT100-no-think,trec,robust-DNA-zero-shot,0,cohen_kappa_binary,0.722443,0.029521
8,qwen3-30B-MT100-no-think,trec,robust-DNA-zero-shot,11,cohen_kappa_binary,0.751498,0.025220
12,qwen3-30B-MT100-no-think,trec,robust-DNA-zero-shot-masked-description-narrative,7,cohen_kappa_binary,0.614420,0.025653
17,gpt-4.1,trec,robust-DNA-zero,0,cohen_kappa_graded,0.360005,0.018068
1,qwen3-14B-MT100-no-think,trec,robust-DNA-zero-shot-masked-description-narrative,0,cohen_kappa_graded,0.347396,0.016438
5,qwen3-14B-MT100-no-think,trec,robust-DNA-zero-shot,0,cohen_kappa_graded,0.449573,0.026192
9,qwen3-30B-MT100-no-think,trec,robust-DNA-zero-shot,11,cohen_kappa_graded,0.469918,0.019984
13,qwen3-30B-MT100-no-think,trec,robust-DNA-zero-shot-masked-description-narrative,7,cohen_kappa_graded,0.351634,0.029276


## Graded Relevance
|Model|Topics|Prompt features|Document scores $\tau$| missing qrels |
|---|---|:---:|---:|---|
|gpt-oss-120b | TREC| — D N A — | 0.45 | 10 |
|gpt-oss-120b | TREC title| — D N A — | 0.44 | 10 |
|qwen3-30B | TREC| — D N A — | 0.5  | 7 |
|qwen3-30B | TREC title| — D N A — | 0.39 | 11 |
|qwen3-14B | TREC| — D N A — | 0.45 | 0 |
|qwen3-14B | TREC title| — D N A — | 0.37 | 0 |
|deepseek-V3.2 | TREC | — D N A — |  0.45 | 18 |
|deepseek-V3.2 | TREC title| — D N A — | 0.42 | 18 |
|deepseek-V3.2 | qwen3-14B | — D N A — |  0.36 |  35 |
|deepseek-V3.2 | deepseek-V2.2 | — D N A — |  0.37 |  3 |


## Binary Relevance
|Model|Topics|Prompt features|Document scores $\tau$| missing qrels |
|---|---|:---:|---:|---|
|gpt-oss-120b | TREC | — D N A — | 0.67 | 10 |
|gpt-oss-120b | TREC title| — D N A — | 0.67 | 10 |
|qwen3-30B| TREC | — D N A — | 0.76 | 7 |
|qwen3-30B | TREC title| — D N A — | 0.65 | 11 |
|qwen3-14B | TREC| — D N A — | 0.72 | 0 |
|qwen3-14B | TREC title| — D N A — | 0.63 | 0 |
|deepseek-V3.2 | TREC | — D N A — | 0.69 | 18 |
|deepseek-V3.2 | TREC title| — D N A — | 0.67 | 18 |
|deepseek-V3.2 | qwen3-14B | — D N A — | 0.54  | 35 |
|deepseek-V3.2 | deepseek-V2.2 | — D N A — |  0.59 | 3 |
|deepseek-V3.2 | deepseek-V2.2 | — D N A — |  0.59 | 3 |